# Hyperparameter Black-Scholes
## Architecture

In [2]:
# Standard library imports
import os
import time
from pathlib import Path

# Data manipulation and mathematics
import numpy as np
import pandas as pd
from scipy.stats import norm
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
from matplotlib import cm

# Deep learning framework (PyTorch)
import torch
import torch.nn as nn
import torch.optim as optim

# Sweep
import itertools

In [3]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/egil10/fys5429.git"
    repo_dir = "/content/fys5429"

    if not os.path.exists(repo_dir):
        !git clone {repo_url} {repo_dir}
    else:
        !git -C {repo_dir} pull

    os.chdir(os.path.join(repo_dir, "code", "notebooks"))

print(f"Working directory: {os.getcwd()}")

Cloning into '/content/fys5429'...
remote: Enumerating objects: 566, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 566 (delta 56), reused 85 (delta 35), pack-reused 454 (from 1)
Receiving objects: 100% (566/566), 61.05 MiB | 34.83 MiB/s, done.
Resolving deltas: 100% (222/222), done.
Working directory: /content/fys5429/code/notebooks


### Colab setup

In [4]:
# Pathways
data_path = Path("..") / "data" / "generated" / "bs_collocation.parquet"

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    out_dir = Path("/content/drive/MyDrive/GITHUB-COLAB/fys5429/code/plots/eda")
else:
    out_dir = Path("..") / "plots" / "eda"

out_dir.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {out_dir}")

Mounted at /content/drive
Plots will be saved to: /content/drive/MyDrive/GITHUB-COLAB/fys5429/code/plots/eda


In [5]:
# Importing BSPINN() Class
import sys
sys.path.insert(0, "../scripts")
from bspinn import BSPINN

ModuleNotFoundError: No module named 'bspinn'

### Global parameters

In [ ]:
# Answer to the universe and everything
torch.manual_seed(42)

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Adding torch backends
torch.backends.cudnn.benchmark = True

# Option Physics
K = 100.0
r = 0.05
sigma = 0.20
T_max = 1.0
S_max = 300.0

# NN defaults (not swept here, just needed for defaults)
HIDDEN_LAYERS = 3
NEURONS_PER_LAYER = 128

# Training
LEARNING_RATE = 0.001
PRINT_ROWS = 20

# PINN Loss Weights
LAMBDA_PDE = 20.0
LAMBDA_IC = 10.0
LAMBDA_BC = 5.0

# Sweep values
LAYERS_VALUES     = [2, 3, 4]
NEURONS_VALUES    = [64, 128, 256]
ACTIVATION_VALUES = ['softplus', 'tanh']
SWEEP_EPOCHS      = 5000

In [ ]:
# Check if data exists and has the right S range, otherwise regenerate
if data_path.exists():
    df_all = pd.read_parquet(data_path)
    s_range = df_all['S'].max()
    if s_range < S_max * 0.9:
        print(f"Existing data has S_max={s_range:.0f}, expected ~{S_max:.0f}. Regenerating...")
        data_path.unlink()

if not data_path.exists():
    print(f"Generating collocation data with S_max={S_max}, T_max={T_max}...")

    N_INTERIOR = 2000
    N_BOUNDARY = 500

    S_interior = torch.rand(N_INTERIOR, 1) * S_max
    tau_interior = torch.rand(N_INTERIOR, 1) * T_max

    S_ic_gen = torch.rand(N_BOUNDARY, 1) * S_max
    tau_ic_gen = torch.zeros(N_BOUNDARY, 1)

    S_bc_gen = torch.zeros(N_BOUNDARY, 1)
    tau_bc_gen = torch.rand(N_BOUNDARY, 1) * T_max

    df_all = pd.concat([
        pd.DataFrame({'S': S_interior.numpy().flatten(),
                       'tau': tau_interior.numpy().flatten(),
                       'point_type': 'interior'}),
        pd.DataFrame({'S': S_ic_gen.numpy().flatten(),
                       'tau': tau_ic_gen.numpy().flatten(),
                       'point_type': 'initial_condition'}),
        pd.DataFrame({'S': S_bc_gen.numpy().flatten(),
                       'tau': tau_bc_gen.numpy().flatten(),
                       'point_type': 'boundary_condition'}),
    ], ignore_index=True)

    data_path.parent.mkdir(parents=True, exist_ok=True)
    df_all.to_parquet(data_path)
    print(f"Saved to {data_path}")
else:
    print(f"Loaded existing data from {data_path} (S_max={df_all['S'].max():.0f})")

# Extract tensors
def extract_tensors(df_subset):
    S_tensor = torch.tensor(df_subset['S'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    tau_tensor = torch.tensor(df_subset['tau'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    return S_tensor, tau_tensor

df_interior = df_all[df_all['point_type'] == 'interior']
S_in, tau_in = extract_tensors(df_interior)

df_ic = df_all[df_all['point_type'] == 'initial_condition']
S_ic, tau_ic = extract_tensors(df_ic)

df_bc = df_all[df_all['point_type'] == 'boundary_condition']
S_bc, tau_bc = extract_tensors(df_bc)

print(f"Interior points: {len(S_in)}")
print(f"IC points:       {len(S_ic)}")
print(f"BC points:       {len(S_bc)}")

In [ ]:
def train_pinn(lambda_pde, lambda_ic, lambda_bc, epochs, lr=0.001,
               hidden_layers=3, neurons=128, activation='softplus', verbose=False):

    model = BSPINN(hidden_layers, neurons, activation=activation).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    grad_ones = torch.ones_like(S_in)

    history = {'epoch': [], 'pde': [], 'ic': [], 'bc': [], 'total': []}
    sample_interval = max(1, epochs // 100)

    for epoch in range(epochs):

        optimizer.zero_grad()

        V_pred = model(S_in, tau_in)

        V_S = torch.autograd.grad(V_pred, S_in, grad_outputs=grad_ones, create_graph=True)[0]
        V_tau = torch.autograd.grad(V_pred, tau_in, grad_outputs=grad_ones, create_graph=True)[0]
        V_SS = torch.autograd.grad(V_S, S_in, grad_outputs=grad_ones, create_graph=True)[0]

        pde_residual = V_tau - (0.5 * sigma**2 * S_in**2 * V_SS + r * S_in * V_S - r * V_pred)
        loss_pde = torch.mean(pde_residual**2)

        V_ic_pred = model(S_ic, tau_ic)
        V_ic_true = torch.relu(S_ic - K)
        loss_ic = torch.mean((V_ic_pred - V_ic_true)**2)

        V_bc_pred = model(S_bc, tau_bc)
        loss_bc = torch.mean((V_bc_pred - 0.0)**2)

        loss = lambda_pde * loss_pde + lambda_ic * loss_ic + lambda_bc * loss_bc
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        if epoch % sample_interval == 0 or epoch == epochs - 1:
            history['epoch'].append(epoch)
            history['pde'].append(loss_pde.item())
            history['ic'].append(loss_ic.item())
            history['bc'].append(loss_bc.item())
            history['total'].append(loss.item())

        if epoch > 500 and epoch % sample_interval == 0:
            if len(history['total']) >= 2 and abs(history['total'][-1] - history['total'][-2]) / (history['total'][-2] + 1e-12) < 1e-5:
                break

    return {
        'model': model,
        'history': history,
        'final_pde': loss_pde.item(),
        'final_ic': loss_ic.item(),
        'final_bc': loss_bc.item(),
        'final_total': loss.item()
    }

In [ ]:
sweep_results = []
start_time = time.time()
combos = list(itertools.product(LAYERS_VALUES, NEURONS_VALUES, ACTIVATION_VALUES))
total_runs = len(combos)

header = f"{'#':>3} | {'Layers':>6} {'Neurons':>7} {'Act':>8} | {'PDE Loss':>12} {'IC Loss':>12} {'BC Loss':>12} | {'Time':>6} {'ETA':>8}"
print(header)
print("─" * len(header))

for i, (layers, neurons, act) in enumerate(combos):
    run_start = time.time()
    result = train_pinn(LAMBDA_PDE, LAMBDA_IC, LAMBDA_BC, SWEEP_EPOCHS,
                        lr=LEARNING_RATE, hidden_layers=layers, neurons=neurons, activation=act)

    result['layers'] = layers
    result['neurons'] = neurons
    result['activation'] = act
    sweep_results.append(result)

    run_sec = time.time() - run_start
    total_elapsed = time.time() - start_time
    eta = (total_elapsed / (i + 1)) * (total_runs - i - 1)

    print(f"{i+1:>3} | {layers:>6} {neurons:>7} {act:>8} | "
          f"{result['final_pde']:>12.6f} {result['final_ic']:>12.6f} {result['final_bc']:>12.6f} | "
          f"{run_sec:>5.0f}s {eta:>6.0f}s")

elapsed = time.time() - start_time
print("─" * len(header))
print(f"Sweep complete: {len(sweep_results)} runs in {int(elapsed//60)}m {int(elapsed%60):02d}s")

In [ ]:
df_sweep = pd.DataFrame([{
    'layers': r['layers'],
    'neurons': r['neurons'],
    'activation': r['activation'],
    'pde_loss': r['final_pde'],
    'ic_loss': r['final_ic'],
    'bc_loss': r['final_bc'],
    'total_loss': r['final_total'],
} for r in sweep_results])

df_sweep = df_sweep.sort_values('pde_loss')
print(df_sweep.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for idx, act in enumerate(ACTIVATION_VALUES):
    df_act = df_sweep[df_sweep['activation'] == act]
    pivot = df_act.pivot_table(index='layers', columns='neurons', values='pde_loss')
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd_r', ax=axes[idx])
    axes[idx].set_title(f'PDE Loss — {act}')

plt.suptitle(f'Architecture Sweep ({SWEEP_EPOCHS} epochs)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(out_dir / "hyper_bs_arch_heatmap.pdf", bbox_inches="tight")
plt.show()

In [ ]:
df_ranked = df_sweep.sort_values('pde_loss').reset_index(drop=True)
labels = [f"{int(r['layers'])}L x {int(r['neurons'])}N ({r['activation']})"
          for _, r in df_ranked.iterrows()]

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(df_ranked)))
bars = ax.barh(range(len(df_ranked)), df_ranked['pde_loss'], color=colors)
ax.set_yticks(range(len(df_ranked)))
ax.set_yticklabels(labels)
ax.set_xlabel('PDE Loss')
ax.set_title(f'All Architectures Ranked by PDE Loss ({SWEEP_EPOCHS} epochs)')
ax.invert_yaxis()

for i, v in enumerate(df_ranked['pde_loss']):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(out_dir / "hyper_bs_arch_ranked.pdf", bbox_inches="tight")
plt.show()

In [ ]:
df_ranked = df_sweep.sort_values('pde_loss').reset_index(drop=True)
top_3 = df_ranked.head(3)

fig, ax = plt.subplots(figsize=(12, 5))

for _, row in top_3.iterrows():
    for orig in sweep_results:
        if (orig['layers'] == row['layers'] and
            orig['neurons'] == row['neurons'] and
            orig['activation'] == row['activation']):
            label = f"{int(row['layers'])}L x {int(row['neurons'])}N ({row['activation']}) — PDE={row['pde_loss']:.4f}"
            ax.plot(orig['history']['epoch'], orig['history']['pde'], label=label)
            break

ax.set_xlabel('Epoch')
ax.set_ylabel('PDE Loss')
ax.set_title(f'Top 3 Architectures — PDE Loss Curves ({SWEEP_EPOCHS} epochs)')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(out_dir / "hyper_bs_arch_top3_curves.pdf", bbox_inches="tight")
plt.show()

In [ ]:
if IN_COLAB:
    import shutil
    for name in ["hyper_bs_arch_heatmap.pdf", "hyper_bs_arch_ranked.pdf", "hyper_bs_arch_top3_curves.pdf"]:
        src = out_dir / name
        dst = Path("/content/drive/MyDrive/GITHUB-COLAB/fys5429/code/plots/eda") / name
        if src.exists():
            shutil.copy2(src, dst)
            print(f"Copied: {name} -> {dst}")